In [1]:
import pandas as pd
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu
import numpy as np

In [2]:
path = r'...' # The path of the file which combines all extracted radiomic features with labels of endoleak and dataset
data = pd.read_excel(path)

In [3]:
def data_statistics(data, data0, data1):
    data_sta = pd.DataFrame(columns = ["Shapiro", "Levene", "Pvalue",
                                       "Mean_0", "SD_0", "Mean_1", "SD_1",  "Mean_A", "SD_A",
                                       "Median_0", "IQR25_0", "IQR75_0",
                                       "Median_1", "IQR25_1", "IQR75_1",
                                       "Median_A", "IQR25_A", "IQR75_A"])
    
    for colName in data.columns[3:]:
        if shapiro(data0[colName])[1] > 0.05 and shapiro(data1[colName])[1] > 0.05:
            if levene(data0[colName], data1[colName])[1] > 0.05:
                sha = 1; lev = 1; Pvalue = ttest_ind(data0[colName], data1[colName])[1]
            else:
                sha = 1; lev = 0; Pvalue = ttest_ind(data0[colName], data1[colName], equal_var=False)[1]
        else:
            sha = 0; lev = 0; Pvalue = mannwhitneyu(data0[colName], data1[colName], alternative='two-sided')[1]
            
        line = pd.Series({"Shapiro":sha, "Levene":lev, "Pvalue":Pvalue,
                          "Mean_0":np.mean(data0[colName]), "SD_0":np.std(data0[colName]),
                          "Mean_1":np.mean(data1[colName]), "SD_1":np.std(data1[colName]),
                          "Mean_A":np.mean(data[colName]), "SD_A":np.std(data[colName]),
                          "Median_0":np.percentile(data0[colName], 50),
                          "IQR25_0":np.percentile(data0[colName], 25), "IQR75_0":np.percentile(data0[colName], 75),
                          "Median_1":np.percentile(data1[colName], 50),
                          "IQR25_1":np.percentile(data1[colName], 25), "IQR75_1":np.percentile(data1[colName], 75),
                          "Median_A":np.percentile(data[colName], 50),
                          "IQR25_A":np.percentile(data[colName], 25), "IQR75_A":np.percentile(data[colName], 75),},
                         name=colName)
        data_sta = data_sta.append(line, ignore_index=False)
        
    return data_sta

In [4]:
def data_selection(data, data0, data1):
    index = []
    
    for colName in data0.columns[3:]:
        if shapiro(data0[colName])[1] > 0.05 and shapiro(data1[colName])[1] > 0.05:
            if levene(data0[colName], data1[colName])[1] > 0.05:
                if ttest_ind(data0[colName], data1[colName])[1] < 0.05:
                    index.append(colName)
            else:
                if ttest_ind(data0[colName], data1[colName], equal_var=False)[1] < 0.05:
                    index.append(colName)
        else:
            if mannwhitneyu(data0[colName], data1[colName], alternative='two-sided')[1] < 0.05:
                index.append(colName)
                
    return index

In [5]:
data_train = data[data['Dataset'] == 'Train']
data0 = data_train[data_train['Endoleak'] == 0]
data1 = data_train[data_train['Endoleak'] == 1]

index = data_selection(data_train, data0, data1)
indexnew = ['ID'] + ['Endoleak'] + ['Dataset'] + index
datanew = data_train[indexnew]
datanew.to_excel('FeatureSelection.xlsx')

In [6]:
data0 = datanew[datanew['Endoleak'] == 0]
data1 = datanew[datanew['Endoleak'] == 1]
datanew_sta = data_statistics(datanew, data0, data1)
datanew_sta.to_excel('FeatureSelection_train_sta.xlsx')